# Import Libraries

In [2]:
import dlt, json, os, time, requests

import pandas as pd
import undetected_chromedriver as uc

from datetime import datetime, timedelta, timezone
from dlt.common import pendulum
from dlt.destinations import filesystem
from dlt.sources.helpers.rest_client import RESTClient
from dlt.sources.helpers.rest_client.auth import BearerTokenAuth
from dlt.sources.helpers.rest_client.paginators import JSONLinkPaginator
from pytz import utc
from selenium import webdriver
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from tqdm.notebook import tqdm
from webdriver_manager.chrome import ChromeDriverManager

# local imports
from scrappers import reddit_scrapper, cryptopanic_api_scrapper


In [2]:
# Define your resource
@dlt.resource(name="crypto_prices", write_disposition="merge", primary_key="timestamp")
def fetch_prices():
    #define the restful client
    client = RESTClient(
        base_url="https://api.coingecko.com/api/v3",
        auth=BearerTokenAuth(token=dlt.secrets['crypto_api_key']['coingecko_token'])
    )

    #define some endpoint params
    crypto_headers = dlt.config['cryptoheaders']

    #check if incremental or initial load
    initial_load = crypto_headers['initial_load']
    if(initial_load is True):
        loading_days = 89
    else:
        loading_days = int(crypto_headers['days'])

    #define the coin id for endpoint parsing
    coin_id = crypto_headers['coin_id']

    #parse the params
    params = {
        "vs_currency": crypto_headers['vs_currency'],
        "days": loading_days
    }
    
    #make request to the endpoint
    response = client.get(f'/coins/{coin_id}/market_chart', params=params)
    print(response)

    #if successful
    if response.status_code==200:
        #print logs
        print(f"Fetching last {params['days']} days of prices for {coin_id} is successful!")
        #parse data
        data = json.loads(str(response.content, 'utf-8'))
        #yield data
        for i in range(len(data["prices"])):
            yield {
                "timestamp": datetime.utcfromtimestamp(data["prices"][i][0] / 1000).replace(tzinfo=utc),
                "price": data["prices"][i][1],
                "market_cap": data["market_caps"][i][1],
                "volume": data["total_volumes"][i][1],
                "asset": coin_id.upper(),
                "ingested_at": datetime.utcnow().replace(tzinfo=utc)
            }
    else:
        print(f"Fetching last {params['days']} days of prices for {coin_id} is not successful due to error {response.status_code}!")


pipeline = dlt.pipeline(
    pipeline_name="crypto_sentiment_pipeline",
    destination='filesystem',
    dataset_name="crypto_data"
)

load_info = pipeline.run(fetch_prices(), loader_file_format="parquet")
print(load_info)


2025-04-09 01:42:15,989|[WARNING]|41960|29088|dlt|utils.py|resolve_merge_strategy:223|Destination does not support any merge strategies and `merge` write disposition  for table `crypto_prices` cannot be met and will fall back to `append`. Change write disposition or try different table format which may offer `merge`: ['delta', 'iceberg'].


<Response [200]>
Fetching last 20 days of prices for bitcoin is successful!


2025-04-09 01:42:16,208|[WARNING]|41960|29088|dlt|utils.py|resolve_merge_strategy:223|Destination does not support any merge strategies and `merge` write disposition  for table `crypto_prices` cannot be met and will fall back to `append`. Change write disposition or try different table format which may offer `merge`: ['delta', 'iceberg'].
2025-04-09 01:42:16,209|[WARNING]|41960|29088|dlt|utils.py|resolve_merge_strategy:223|Destination does not support any merge strategies and `merge` write disposition  for table `crypto_prices` cannot be met and will fall back to `append`. Change write disposition or try different table format which may offer `merge`: ['delta', 'iceberg'].
2025-04-09 01:42:17,222|[WARNING]|41960|29088|dlt|utils.py|resolve_merge_strategy:223|Destination does not support any merge strategies and `merge` write disposition  for table `crypto_prices` cannot be met and will fall back to `append`. Change write disposition or try different table format which may offer `merge`:

Pipeline crypto_sentiment_pipeline load step completed in 1.98 seconds
1 load package(s) were loaded to destination filesystem and into dataset crypto_data
The filesystem destination used gs://zoomcamp_project_v1 location to store data
Load package 1744134135.4769204 is LOADED and contains no failed jobs


In [2]:
# Instantiate the Reddit scraper
scraper = reddit_scrapper(headless=False, days_extracted=5)

# Step 1: Extract posts
scraper.get_posts()

# Step 2: Define the DLT resource
@dlt.resource(write_disposition="append", primary_key="comment_id", name="crypto_sentiments")
def extract_reddit_comments():
    yield from scraper.get_comments()

# Step 3: Create the DLT pipeline
pipeline = dlt.pipeline(
    pipeline_name="crypto_sentiment_pipeline",
    destination="filesystem",
    dataset_name="crypto_sentiments"
)

# Step 4: Run pipeline — pass the resource, not the function call
load_info = pipeline.run(extract_reddit_comments, loader_file_format="parquet")
print(load_info)

# Step 5: Close the browser after scraping
scraper.close()

  0%|          | 0/49 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

  0%|          | 0/85 [00:00<?, ?it/s]

  0%|          | 0/75 [00:00<?, ?it/s]

  0%|          | 0/262 [00:00<?, ?it/s]

...

  0%|          | 0/49 [00:00<?, ?it/s]

  0%|          | 0/31 [00:00<?, ?it/s]

  0%|          | 0/19 [00:00<?, ?it/s]

2025-04-10 08:27:09,911|[WARNING]|126|140630821284736|dlt|utils.py|resolve_merge_strategy:260|Destination does not support any merge strategies and `merge` write disposition  for table `crypto_prices` cannot be met and will fall back to `append`. Change write disposition or try different table format which may offer `merge`: ['delta', 'iceberg'].


Pipeline crypto_sentiment_pipeline load step completed in 2.06 seconds
1 load package(s) were loaded to destination filesystem and into dataset crypto_sentiments
The filesystem destination used gs://zoomcamp_project_v1 location to store data
Load package 1744273215.5220673 is LOADED and contains no failed jobs


In [4]:
# Instantiate the Reddit scraper
scraper = cryptopanic_api_scrapper(headless=False, days_extracted=100)

# Step 1: Extract the crypto posts
scraper.fetch_cryptopanic_feed()

# Step 2: Define the DLT resource
@dlt.resource(write_disposition="append", primary_key="comment_id", name="cryptopanic_sentiments")
def extract_cryptopanic_comments():
    yield from scraper.get_comments()

# Step 3: Create the DLT pipeline
pipeline = dlt.pipeline(
    pipeline_name="crypto_sentiment_pipeline",
    destination="filesystem",
    dataset_name="crypto_sentiments"
)

# Step 4: Run pipeline — pass the resource, not the function call
load_info = pipeline.run(extract_cryptopanic_comments, loader_file_format="parquet")
print(load_info)

# Step 5: Close the browser after scraping
scraper.close()

https://cryptopanic.com/api/v1/posts/
https://cryptopanic.com/api/v1/posts/?auth_token=1d51e9d443bbcdc78a7d16e493d1b0223c81ea36&currencies=BTC&public=true&filter=hot&page=2
https://cryptopanic.com/api/v1/posts/?auth_token=1d51e9d443bbcdc78a7d16e493d1b0223c81ea36&currencies=BTC&public=true&filter=hot&page=3
https://cryptopanic.com/api/v1/posts/?auth_token=1d51e9d443bbcdc78a7d16e493d1b0223c81ea36&currencies=BTC&public=true&filter=hot&page=4
https://cryptopanic.com/api/v1/posts/?auth_token=1d51e9d443bbcdc78a7d16e493d1b0223c81ea36&currencies=BTC&public=true&filter=hot&page=5
https://cryptopanic.com/api/v1/posts/?auth_token=1d51e9d443bbcdc78a7d16e493d1b0223c81ea36&currencies=BTC&public=true&filter=hot&page=6
https://cryptopanic.com/api/v1/posts/?auth_token=1d51e9d443bbcdc78a7d16e493d1b0223c81ea36&currencies=BTC&public=true&filter=hot&page=7
https://cryptopanic.com/api/v1/posts/?auth_token=1d51e9d443bbcdc78a7d16e493d1b0223c81ea36&currencies=BTC&public=true&filter=hot&page=8
https://cryptopan

  0%|          | 0/182 [00:00<?, ?it/s]

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
❌ Error extracting comments: Message: no such element: Unable to locate element: {"method":"css selector","selector":".comment-body"}
  (Session info: chrome=135.0.7049.84); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
#0 0x55abb8a18d0a <unknown>
#1 0x55abb84c95f0 <unknown>
#2 0x55abb851aa33 <unknown>
#3 0x55abb851ac21 <unknown>
#4 0x55abb850e5c6 <unknown>
#5 0x55abb854068d <unknown>
#6 0x55abb850e4ba <unknown>
#7 0x55abb854082e <unknown>
#8 0x55abb8566660 <unknown>
#9 0x55abb8540433 <unknown>
#10 0x55abb850cea3 <unknown>
#11 0x55abb850db01 <unknown>
#12 0x55abb89ddb5b <unknown>
#13 0x55abb89e1a41 <unknown>
#14 0x55abb89c4c52 <unknown>
#15 0x55abb89e25b4 <unknown>
#16 0x55abb89a8f0f <unknown>
#17 0x55abb8a06db8 <unknown>
#18 0x55abb8a06f96 <unknown>
#19 0x55abb8a17b56 <unknown>
#20 0x7f7988b481f5 <unknown>

17
18
19
20
21
22
23
24
25
26


PipelineStepFailed: Pipeline execution failed at stage extract when processing package 1744280958.1467388 with exception:

<class 'TypeError'>
Type is not JSON serializable: WebElement